# 🎒 Desi CS Student Survival — Quickstart & EDA
Synthetic data about life as an Indian CS student: hostel, mess, bunking, backlogs, LeetCode guilt and placements.
This notebook tours the data and runs a baseline for each classic ML task: **regression, classification, clustering, PCA and NLP**.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
D = pd.read_csv('/kaggle/input/desi-cs-student-survival/students.csv')
S = pd.read_csv('/kaggle/input/desi-cs-student-survival/semester_records.csv')
print(D.shape, S.shape); D.head()

## 1. Missing data — on purpose
Three kinds: random (survey skips), structural (mess columns for day scholars, placement for non-final-years) and MAR (low-CGPA students hide their CGPA).

In [ ]:
m = D.isna().mean().sort_values(ascending=False); m[m>0].plot.barh(figsize=(7,5), color='#e4572e'); plt.title('Share missing'); plt.show()

## 2. The semester 3–4 slump (DSA/OS/DBMS reality check) and senioritis

In [ ]:
S.groupby('semester').sgpa.mean().plot(marker='o', figsize=(7,3.5)); plt.ylabel('mean SGPA'); plt.title('Average SGPA by semester'); plt.show()

## 3. Hostel life

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
sns.boxplot(data=D,x='residence',y='maggi_per_week',ax=ax[0]); sns.boxplot(data=D,x='residence',y='wifi_speed_mbps',ax=ax[1]); ax[1].set_yscale('log')
sns.scatterplot(data=D.sample(1500,random_state=0),x='mess_rating',y='food_poisoning_incidents',alpha=.3,ax=ax[2])
for a in ax[:2]: a.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.show()

## 4. What drives placement? (final-years only)

In [ ]:
F = D[D.year_of_study==4].copy(); F['placed_bin']=(F.placed=='Yes').astype(int)
print(F.groupby('college_tier').placed_bin.mean().round(2).to_dict())
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
F.groupby(pd.cut(F.cgpa,[0,6,7,8,10])).placed_bin.mean().plot.bar(ax=ax[0],title='Placement rate by CGPA'); ax[0].set_xlabel('')
sns.histplot(F.package_lpa.dropna(),bins=40,ax=ax[1]).set(title='Package (LPA) is right-skewed'); plt.tight_layout(); plt.show()

## 5. Baselines

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, roc_auc_score
PLACE=['placed','placement_category','offers_received','package_lpa']; TXT=['mess_review_text','reason_for_last_bunk']; IDL=['student_id','archetype']
def prep(X):
    cat=X.select_dtypes(include=['object','bool']).columns.tolist(); num=[c for c in X.columns if c not in cat]
    return ColumnTransformer([('n',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),
                              ('c',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat)])
# Regression: CGPA
r=D.dropna(subset=['cgpa']).drop(columns=PLACE+TXT+IDL+['backlogs']); X,y=r.drop(columns='cgpa'),r.cgpa
a,b,c,d=train_test_split(X,y,random_state=42)
print('CGPA  LinReg R2 =',round(r2_score(d,make_pipeline(prep(X),LinearRegression()).fit(a,c).predict(b)),3))
# Classification: placed?
X=F.drop(columns=PLACE+TXT+IDL+['placed_bin']); y=F.placed_bin
a,b,c,d=train_test_split(X,y,random_state=42,stratify=y)
p=make_pipeline(prep(X),LogisticRegression(max_iter=1000)).fit(a,c); print('Placed LogReg AUC =',round(roc_auc_score(d,p.predict_proba(b)[:,1]),3))

## 6. Clustering — can K-Means find the 7 hidden personas?

In [ ]:
from sklearn.cluster import KMeans; from sklearn.decomposition import PCA; from sklearn.metrics import adjusted_rand_score
cols=['attendance_pct','study_hours_daily','gaming_hours_weekly','sleep_hours','leetcode_solved','communication_skill','club_memberships','linkedin_connections','certifications_count','chai_cups_per_day','bunked_classes_per_month','cgpa']
Z=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(np.log1p(D[cols].clip(lower=0))))
km=KMeans(7,n_init=10,random_state=42).fit(Z); print('ARI =',round(adjusted_rand_score(D.archetype,km.labels_),3))
P=PCA(2).fit_transform(Z); fig,ax=plt.subplots(1,2,figsize=(13,4.5))
sns.scatterplot(x=P[:,0],y=P[:,1],hue=D.archetype,s=8,ax=ax[0]).set(title='PCA coloured by TRUE persona')
sns.scatterplot(x=P[:,0],y=P[:,1],hue=km.labels_,palette='tab10',s=8,ax=ax[1]).set(title='PCA coloured by K-Means'); plt.tight_layout(); plt.show()

## 7. NLP — mess review → rating

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer; from sklearn.linear_model import Ridge
t=D.dropna(subset=['mess_review_text']); a,b,c,d=train_test_split(t.mess_review_text,t.mess_rating,random_state=42)
p=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2),Ridge()).fit(a,c); print('R2 =',round(r2_score(d,p.predict(b)),3))
print(t.sample(3,random_state=1)[['mess_rating','mess_review_text']].to_string())

## Your turn
* Beat my K-Means ARI. Try GMM, UMAP, DBSCAN.
* Does a model use `gender`? (It shouldn't — the generator has no effect.) Audit it.
* Use `semester_records.csv` for per-student trends.
* Warning: `package_lpa` leaks `placed`. 😉

Docs, generator and baselines: https://github.com/Pranav-Singh-Devloper/desi-cs-student-survival-dataset